In [1]:
from pathlib import Path
import sys, platform, json

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find pyproject.toml. Open Jupyter inside the project.")

PROJECT_ROOT = find_project_root(Path.cwd())
SMALL_LMDB = PROJECT_ROOT / "data/raw/small"
LARGE_LMDB = PROJECT_ROOT / "data/raw/large"

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Small LMDB exists:", SMALL_LMDB.exists(), SMALL_LMDB)
print("Large LMDB exists:", LARGE_LMDB.exists(), LARGE_LMDB)

import torch
import lmdb
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("LMDB module:", lmdb.__file__)

Project root: /Users/thinyadanaroo/sch_projects/PROM02/PROM02_240291316/doc-forgery-detection
Python: 3.12.4 (v3.12.4:8e8a4baf65, Jun  6 2024, 17:33:18) [Clang 13.0.0 (clang-1300.0.29.30)]
Platform: macOS-26.6.2-arm64-arm-64bit
Small LMDB exists: True /Users/thinyadanaroo/sch_projects/PROM02/PROM02_240291316/doc-forgery-detection/data/raw/small
Large LMDB exists: True /Users/thinyadanaroo/sch_projects/PROM02/PROM02_240291316/doc-forgery-detection/data/raw/large
PyTorch: 2.3.1
MPS available: True
LMDB module: /Users/thinyadanaroo/sch_projects/PROM02/PROM02_240291316/doc-forgery-detection/.venv/lib/python3.12/site-packages/lmdb/__init__.py


In [2]:
def open_lmdb(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return lmdb.open(str(path), readonly=True, lock=False,
                     readahead=False, meminit=False, max_readers=512)

def inspect_lmdb(path, name):
    env = open_lmdb(path)
    try:
        with env.begin() as txn:
            metadata = txn.get(b"num-samples")
            counts = {"images": 0, "labels": 0, "other": 0}
            for key, _ in txn.cursor():
                if key.startswith(b"image-"):
                    counts["images"] += 1
                elif key.startswith(b"label-"):
                    counts["labels"] += 1
                else:
                    counts["other"] += 1
        result = {
            "dataset": name,
            "metadata_num_samples": int(metadata) if metadata else None,
            **counts
        }
        print(json.dumps(result, indent=2))
        return result
    finally:
        env.close()

small_summary = inspect_lmdb(SMALL_LMDB, "small")
large_summary = inspect_lmdb(LARGE_LMDB, "large")

{
  "dataset": "small",
  "metadata_num_samples": 2000,
  "images": 2000,
  "labels": 2000,
  "other": 1
}
{
  "dataset": "large",
  "metadata_num_samples": 120000,
  "images": 120000,
  "labels": 120000,
  "other": 1
}


In [ ]:
def read_pair(txn, index):
    image_bytes = txn.get(f"image-{index:09d}".encode())
    mask_bytes = txn.get(f"label-{index:09d}".encode())
    if image_bytes is None or mask_bytes is None:
        raise KeyError(f"Missing image or mask for index {index}")
    image = Image.open(BytesIO(image_bytes)).convert("RGB")
    mask = Image.open(BytesIO(mask_bytes)).convert("L")
    return image, mask

def inspect_samples(path, indices, name):
    env = open_lmdb(path)
    rows = []
    try:
        with env.begin() as txn:
            for i in indices:
                image, mask = read_pair(txn, i)
                arr = np.asarray(mask)
                row = {
                    "dataset": name, "index": i,
                    "image_size": image.size, "mask_size": mask.size,
                    "mask_values": np.unique(arr).tolist(),
                    "foreground_pixels_mask_gt_zero": int((arr > 0).sum())
                }
                rows.append(row)
                print(row)
    finally:
        env.close()
    return rows

small_samples = inspect_samples(SMALL_LMDB, [0, 1, 10, 100, 1999], "small")
large_samples = inspect_samples(
    LARGE_LMDB, [0, 1, 10, 100, 1000, 10000, 50000, 100000, 119999], "large"
)

In [ ]:
def show_pair(path, index, name):
    env = open_lmdb(path)
    try:
        with env.begin() as txn:
            image, mask = read_pair(txn, index)
    finally:
        env.close()

    image_arr = np.asarray(image)
    mask_arr = np.asarray(mask)
    foreground = mask_arr > 0
    overlay = image_arr.copy()
    overlay[foreground] = (
        0.5 * overlay[foreground] + 0.5 * np.array([255, 0, 0])
    ).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image_arr); axes[0].set_title(f"{name} image #{index}")
    axes[1].imshow(foreground, cmap="gray"); axes[1].set_title("Mask (mask > 0)")
    axes[2].imshow(overlay); axes[2].set_title("Overlay")
    for ax in axes: ax.axis("off")
    plt.tight_layout()
    plt.show()

show_pair(LARGE_LMDB, 0, "large")
show_pair(LARGE_LMDB, 100, "large")

In [ ]:
from collections import Counter

def verify_dataset(path, expected_samples, name):
    env = open_lmdb(path)
    errors, empty_masks = [], 0
    image_sizes, mask_sizes = Counter(), Counter()
    mask_values = set()
    try:
        with env.begin() as txn:
            for i in range(expected_samples):
                try:
                    image, mask = read_pair(txn, i)
                    image_sizes[image.size] += 1
                    mask_sizes[mask.size] += 1
                    arr = np.asarray(mask)
                    mask_values.update(np.unique(arr).tolist())
                    if not np.any(arr):
                        empty_masks += 1
                    if image.size != mask.size:
                        errors.append(f"{i}: image/mask dimensions differ")
                except Exception as exc:
                    errors.append(f"{i}: {type(exc).__name__}: {exc}")
                if (i + 1) % 10000 == 0:
                    print(f"Checked {i+1:,}/{expected_samples:,}")
    finally:
        env.close()

    result = {
        "dataset": name, "expected_samples": expected_samples,
        "errors": len(errors), "empty_masks": empty_masks,
        "image_dimensions": dict(image_sizes),
        "mask_dimensions": dict(mask_sizes),
        "mask_values": sorted(mask_values),
        "first_errors": errors[:20]
    }
    print(json.dumps(result, indent=2))
    return result

# small_verification = verify_dataset(SMALL_LMDB, 2000, "small")
# large_verification = verify_dataset(LARGE_LMDB, 120000, "large")